# Datathon Passos Mágicos - Análise Exploratória e Storytelling

Este notebook responde às 11 perguntas do desafio com base na base consolidada analítica e, quando necessário, na base reduzida para modelagem.

## Objetivos
- Entender a evolução dos indicadores pedagógicos, psicossociais e de engajamento;
- Identificar padrões de risco de defasagem;
- Avaliar a efetividade do programa ao longo dos anos e fases;
- Preparar insumos para o modelo preditivo e para o storytelling da apresentação final.

## Bases utilizadas
- Base analítica consolidada: `base_PEDE_consolidada_analitica.parquet`
- Base reduzida para ML: `base_processada_reduzida_ML.parquet`

In [ ]:
# 2) Imports e configuraçãoimport warnings
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    RocCurveDisplay
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)

sns.set_theme(style="whitegrid")

In [ ]:
# 3) Carga das bases
df = pd.read_parquet("../data/processed/base_PEDE_consolidada_analitica.parquet")
df_ml = pd.read_parquet("../data/processed/base_processada_reduzida_ML.parquet")

print("Base analítica:", df.shape)
print("Base ML:", df_ml.shape)

display(df.head())
display(df_ml.head())


In [ ]:
# 4) Padronizações auxiliares
# cria coluna inde única na base analítica
df["inde"] = np.nan

if "inde_2022" in df.columns:
    df.loc[df["ano_pede"] == 2022, "inde"] = df["inde_2022"]
if "inde_2023" in df.columns:
    df.loc[df["ano_pede"] == 2023, "inde"] = df["inde_2023"]
if "inde_2024" in df.columns:
    df.loc[df["ano_pede"] == 2024, "inde"] = df["inde_2024"]

# garantir numéricos
cols_numericas = [
    "idade", "ano_ingresso", "inde", "iaa", "ieg", "ips", "ipp",
    "ida", "mat", "por", "ing", "ipv", "ian", "defasagem"
]

for col in cols_numericas:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# categorias úteis
for col in ["fase", "turma", "genero", "instituicao_ensino", "pedra_2022", "pedra_2023", "pedra_2024"]:
    if col in df.columns:
        df[col] = df[col].astype("string").str.strip()

df.info()

In [ ]:
# 5) Funções auxiliares
def plot_bar(data, x, y, title, figsize=(10,5), rotation=0):
    plt.figure(figsize=figsize)
    sns.barplot(data=data, x=x, y=y)
    plt.title(title)
    plt.xticks(rotation=rotation)
    plt.tight_layout()
    plt.show()


def plot_line(data, x, y, hue=None, title="", figsize=(10,5), rotation=0, marker="o"):
    plt.figure(figsize=figsize)
    sns.lineplot(data=data, x=x, y=y, hue=hue, marker=marker)
    plt.title(title)
    plt.xticks(rotation=rotation)
    plt.tight_layout()
    plt.show()


def plot_heatmap_corr(df_corr, title="Mapa de Correlação"):
    plt.figure(figsize=(10, 7))
    sns.heatmap(df_corr, annot=True, cmap="Blues", fmt=".2f")
    plt.title(title)
    plt.tight_layout()
    plt.show()


def categorizar_defasagem(valor):
    if pd.isna(valor):
        return "Sem informação"
    if valor <= 0:
        return "Sem defasagem"
    elif valor == 1:
        return "Defasagem leve"
    elif valor == 2:
        return "Defasagem moderada"
    else:
        return "Defasagem severa"

### 3. Engajamento nas atividades (IEG)

In [ ]:
# 14) Correlação entre IEG, IDA e IPV
corr_cols = [c for c in ["ieg", "ida", "ipv"] if c in df.columns]
corr_ieg = df[corr_cols].corr()
display(corr_ieg)

plot_heatmap_corr(corr_ieg, "Correlação entre IEG, IDA e IPV")

In [ ]:
# 15) Dispersão IEG x IDA
plt.figure(figsize=(8,6))
sns.scatterplot(data=df, x="ieg", y="ida", hue="ano_pede", alpha=0.7)
plt.title("Relação entre engajamento (IEG) e desempenho acadêmico (IDA)")
plt.tight_layout()
plt.show()

In [ ]:
# 16) Dispersão IEG x IPV
plt.figure(figsize=(8,6))
sns.scatterplot(data=df, x="ieg", y="ipv", hue="ano_pede", alpha=0.7)
plt.title("Relação entre engajamento (IEG) e ponto de virada (IPV)")
plt.tight_layout()
plt.show()

### Leitura analítica
- Esta seção mostra quantos alunos estão sem defasagem, com defasagem leve, moderada ou severa.
- A evolução por ano ajuda a verificar se o programa está conseguindo reduzir casos moderados e severos.
- A análise por fase permite identificar em quais etapas da jornada os alunos concentram maior vulnerabilidade.

### 2. Desempenho acadêmico (IDA)

In [ ]:
# 10) IDA médio por ano
ida_ano = (
    df.groupby("ano_pede", as_index=False)["ida"]
      .mean()
      .rename(columns={"ida": "ida_medio"})
)

display(ida_ano)

plot_line(
    ida_ano,
    x="ano_pede",
    y="ida_medio",
    title="Evolução do IDA médio por ano"
)

In [ ]:
#11) IDA médio por fase e ano
if "fase" in df.columns:
    ida_fase_ano = (
        df.groupby(["ano_pede", "fase"], as_index=False)["ida"]
          .mean()
          .rename(columns={"ida": "ida_medio"})
    )

    display(ida_fase_ano.head(20))

    plot_line(
        ida_fase_ano,
        x="ano_pede",
        y="ida_medio",
        hue="fase",
        title="IDA médio por fase ao longo dos anos",
        figsize=(12,6)
    )

In [ ]:
#17) Quartis de engajamento
df["faixa_ieg"] = pd.qcut(df["ieg"], q=4, labels=["Baixo", "Médio-baixo", "Médio-alto", "Alto"])

resumo_ieg = (
    df.groupby("faixa_ieg", observed=False)[["ida", "ipv"]]
      .mean()
      .reset_index()
)

display(resumo_ieg)

### Leitura analítica
- Correlação positiva entre IEG e IDA sugere que maior engajamento tende a acompanhar melhor desempenho acadêmico.
- Correlação positiva entre IEG e IPV indica que alunos mais engajados têm maior chance de apresentar ponto de virada favorável.
- O uso de quartis facilita traduzir o achado para o storytelling executivo.